# HydrAI: Generalized Plug Flow Reactor Simulation Framework

*A comprehensive computational framework for modeling steam cracking reactions in plug flow reactors using Cantera*

## Executive Summary

This interactive computational notebook provides a systematic approach to modeling steam cracking processes in plug flow reactors (PFRs). The framework supports multiple hydrocarbon feedstocks with dedicated kinetic mechanisms and enables comprehensive analysis of reaction kinetics, heat transfer, and fluid dynamics.

## Supported Feedstocks

| Feedstock | Chemical Formula | Primary Products | Mechanism Complexity |
|-----------|------------------|------------------|---------------------|
| **Ethane** | C₂H₆ | Ethylene, Hydrogen | 35 species, 135 reactions |
| **Propane** | C₃H₈ | Ethylene, Propylene, Hydrogen | 53 species, 325 reactions |
| **Naphtha** | Mixed C₅-C₁₂ | Ethylene, Propylene, Aromatics | 1,951 species, 82,557 reactions |
| **n-Hexane** | C₆H₁₄ | Ethylene, Propylene, Butenes | 153 species, 2,146 reactions |

## Technical Capabilities

- **Multi-scale Modeling**: Integration of reaction kinetics, heat transfer, and fluid dynamics
- **Mechanism Management**: Automated handling of complex kinetic mechanisms
- **Parameter Sensitivity**: Systematic analysis of operating conditions
- **Data Export**: Professional-grade results formatting and visualization
- **Reproducibility**: Comprehensive configuration documentation

---

**Principal Investigator:** Nikolas Karefyllidis, PhD  
**Institution:** Chemical Engineering Research Group  
**Development Period:** 2025-01-15 to Present  
**Software License:** MIT Open Source


## 1. Computational Environment Setup

Initialize the computational environment by importing required scientific computing libraries and configuring visualization parameters.


In [ ]:
# =============================================================================
# SCIENTIFIC COMPUTING LIBRARIES
# =============================================================================
# Chemical kinetics and thermodynamics
import cantera as ct

# Numerical computing and data analysis
import numpy as np
import scipy.integrate
import pandas as pd

# Scientific visualization
import matplotlib.pyplot as plt
import matplotlib.style as style

# =============================================================================
# SYSTEM UTILITIES AND CONFIGURATION
# =============================================================================
import os
import time
import json
from datetime import datetime
import warnings

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings('ignore')

# =============================================================================
# VISUALIZATION CONFIGURATION
# =============================================================================
# Configure matplotlib for publication-quality plots
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['grid.alpha'] = 0.3

# =============================================================================
# ENVIRONMENT VERIFICATION
# =============================================================================
print("=" * 60)
print("COMPUTATIONAL ENVIRONMENT VERIFICATION")
print("=" * 60)
print(f"- Cantera Version: {ct.__version__}")
print(f"- NumPy Version: {np.__version__}")
print(f"- Pandas Version: {pd.__version__}")
print(f"- Matplotlib Version: {plt.matplotlib.__version__}")
print("- All required libraries successfully imported")
print("=" * 60)


: 

## 2. Chemical Database Initialization

Load the comprehensive chemical database containing thermodynamic properties, kinetic mechanisms, and process parameters for supported hydrocarbon feedstocks.


In [ ]:
# =============================================================================
# CHEMICAL DATABASE LOADING
# =============================================================================
# Load comprehensive chemical database from JSON configuration file
with open('reactant_database.json', 'r') as f:
    database = json.load(f)

# =============================================================================
# DATABASE VERIFICATION AND DISPLAY
# =============================================================================
print("=" * 80)
print("CHEMICAL DATABASE VERIFICATION")
print("=" * 80)
print(f"Database contains {len(database['reactants'])} supported feedstocks:")
print()

for key, reactant in database['reactants'].items():
    print(f"Feedstock: {reactant['name']} ({reactant['formula']})")
    print(f"  - Kinetic Mechanism: {reactant['mechanism_file']}")
    
    # Extract species count from mechanism filename
    mechanism_file = reactant['mechanism_file']
    if 'species_' in mechanism_file:
        species_count = mechanism_file.split('species_')[1].split('.')[0]
        print(f"  - Chemical Species: {species_count}")
    
    print(f"  - Process Description: {reactant['description']}")
    print()

print("=" * 80)
print("DATABASE LOADING COMPLETED SUCCESSFULLY")
print("=" * 80)


## 3. Dynamic Configuration Management

Implementation of automated configuration generation system that creates complete simulation parameter sets based on feedstock selection and process requirements.


In [ ]:
def generate_configuration(reactant_key, database):
    """
    Generate comprehensive simulation configuration for specified feedstock.
    
    This function creates a complete configuration dictionary by dynamically
    replacing template placeholders with feedstock-specific parameters and
    validating kinetic mechanism properties.
    
    Parameters:
    -----------
    reactant_key : str
        Unique identifier for the feedstock in the chemical database
    database : dict
        Complete chemical database containing feedstock properties
        
    Returns:
    --------
    dict
        Complete configuration dictionary with validated parameters
        
    Raises:
    -------
    ValueError
        If specified reactant is not found in the database
    FileNotFoundError
        If required configuration template is missing
        
    Notes:
    ------
    The function performs automatic validation of kinetic mechanisms
    and updates species/reaction counts based on actual mechanism files.
    """
    # Validate reactant existence in database
    if reactant_key not in database['reactants']:
        raise ValueError(f"Feedstock '{reactant_key}' not found in chemical database")
    
    reactant_info = database['reactants'][reactant_key]
    
    # Load configuration template
    try:
        with open('config_template.json', 'r') as f:
            config_str = f.read()
    except FileNotFoundError:
        raise FileNotFoundError("Configuration template file not found")
    
    # Replace template placeholders with feedstock-specific data
    config_str = config_str.replace('{{REACTANT_NAME}}', reactant_info['name'])
    config_str = config_str.replace('{{REACTANT_DESCRIPTION}}', reactant_info['description'])
    config_str = config_str.replace('{{REACTANT_KEY}}', reactant_key)
    config_str = config_str.replace('{{MECHANISM_FILE}}', reactant_info['mechanism_file'])
    config_str = config_str.replace('{{DATE}}', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
    
    # Generate feed composition string from database parameters
    feed_composition = f"{reactant_info['feed_species']}, {reactant_info['diluent']}:{reactant_info['diluent_ratio']}"
    config_str = config_str.replace('{{FEED_COMPOSITION}}', feed_composition)
    
    # Validate kinetic mechanism and extract actual species/reaction counts
    try:
        gas = ct.Solution(reactant_info['mechanism_file'])
        n_species = gas.n_species
        n_reactions = gas.n_reactions
    except Exception as e:
        print(f"Warning: Kinetic mechanism validation failed: {e}")
        print("Using database values as fallback")
        n_species = reactant_info.get('n_species', 'Unknown')
        n_reactions = reactant_info.get('n_reactions', 'Unknown')
    
    # Update configuration with validated mechanism data
    config_str = config_str.replace('{{N_SPECIES}}', str(n_species))
    config_str = config_str.replace('{{N_REACTIONS}}', str(n_reactions))
    
    # Parse and return complete configuration
    return json.loads(config_str)

# =============================================================================
# CONFIGURATION MANAGEMENT SYSTEM VERIFICATION
# =============================================================================
print("=" * 60)
print("CONFIGURATION MANAGEMENT SYSTEM INITIALIZED")
print("=" * 60)
print("- Dynamic configuration generation function loaded")
print("- Template-based parameter replacement system active")
print("- Kinetic mechanism validation enabled")
print("=" * 60)


## 4. Simulation Component Initialization

Implementation of core initialization functions for kinetic mechanisms, initial conditions, and heat transfer profiles required for PFR simulation.


In [ ]:
def setup_mechanism(config):
    """
    Initialize Cantera gas phase object with kinetic mechanism.
    
    Loads and validates the kinetic mechanism file specified in the configuration,
    creating a Cantera Solution object for thermodynamic and kinetic calculations.
    
    Parameters:
    -----------
    config : dict
        Simulation configuration containing mechanism file path
        
    Returns:
    --------
    ct.Solution
        Cantera gas phase object with loaded kinetic mechanism
        
    Notes:
    ------
    The function performs automatic validation of the mechanism file
    and reports species and reaction counts for verification.
    """
    mechanism_file = config['mechanism']['reaction_mechanism_file']
    gas = ct.Solution(mechanism_file)
    
    print(f"Kinetic mechanism loaded: {mechanism_file}")
    print(f"Chemical species: {gas.n_species}")
    print(f"Elementary reactions: {gas.n_reactions}")
    
    return gas

def setup_initial_conditions(gas, config):
    """
    Initialize reactor operating conditions and geometric parameters.
    
    Sets up the initial thermodynamic state of the gas phase and calculates
    reactor geometry parameters required for mass and momentum balances.
    
    Parameters:
    -----------
    gas : ct.Solution
        Cantera gas phase object with loaded mechanism
    config : dict
        Simulation configuration containing operating conditions
        
    Returns:
    --------
    tuple
        (T_0, p_0, composition_0, length, diameter, area, roughness, mass_flow_rate, u_0)
        Initial temperature, pressure, composition, and reactor parameters
    """
    # Extract initial thermodynamic conditions
    composition_0 = config['initial_conditions']['composition']
    T_0 = config['initial_conditions']['temperature_K']
    p_0 = config['initial_conditions']['pressure_Pa']
    
    # Set initial gas phase state
    gas.TPY = T_0, p_0, composition_0
    
    # Extract reactor geometry parameters
    length = config['reactor_geometry']['length_m']
    diameter = config['reactor_geometry']['diameter_m']
    area = np.pi * (diameter / 2) ** 2  # Cross-sectional area [m²]
    roughness = config['reactor_geometry']['roughness_m']
    
    # Calculate initial velocity from mass flow rate
    mass_flow_rate = config['operating_conditions']['mass_flow_rate_kgps']
    u_0 = mass_flow_rate / (gas.density * area)  # Initial velocity [m/s]
    
    return T_0, p_0, composition_0, length, diameter, area, roughness, mass_flow_rate, u_0

def setup_heat_flux(config):
    """
    Initialize heat flux profile for reactor wall heat transfer.
    
    Loads spatial heat flux profile from JSON configuration file and creates
    a Cantera interpolation function for heat transfer calculations.
    
    Parameters:
    -----------
    config : dict
        Simulation configuration containing heat flux file path
        
    Returns:
    --------
    tuple
        (hf, z_profile, heatflux_profile)
        Heat flux function, position array, and heat flux array
    """
    heat_flux_file = config['mechanism']['heat_flux_file']
    
    # Load heat flux profile data from JSON file
    with open(heat_flux_file, 'r') as f:
        heat_flux_data = json.load(f)
    
    # Extract spatial heat flux data points
    data_points = heat_flux_data['heat_flux_profile']['data_points']
    z_profile = np.array([point['position'] for point in data_points])
    heatflux_profile = np.array([point['heat_flux'] for point in data_points])
    
    # Create Cantera interpolation function for heat flux
    hf = ct.Func1(lambda z: np.interp(z, z_profile, heatflux_profile))
    
    return hf, z_profile, heatflux_profile

# =============================================================================
# SIMULATION COMPONENT INITIALIZATION VERIFICATION
# =============================================================================
print("=" * 60)
print("SIMULATION COMPONENT INITIALIZATION COMPLETED")
print("=" * 60)
print("- Kinetic mechanism setup function loaded")
print("- Initial conditions setup function loaded")
print("- Heat flux profile setup function loaded")
print("=" * 60)


## 5. Pressure Drop Calculation

Function to calculate pressure drop using the Churchill correlation.


In [ ]:
def calculate_pressure_drop(gas, u, diameter, roughness=0.0):
    """Calculate pressure drop using Churchill correlation."""
    # Fluid properties
    rho = gas.density
    mu = gas.viscosity
    Re = rho * u * diameter / mu
    
    # Churchill correlation for friction factor
    A = (2.457 * np.log(1 / ((7/Re)**0.9 + 0.27 * roughness/diameter)))**16
    B = (37530/Re)**16
    f = 8 * ((8/Re)**12 + 1/(A + B)**1.5)**(1/12)
    
    # Pressure drop per unit length
    dpdz = -f * rho * u**2 / (2 * diameter)
    
    return dpdz

print("Pressure drop calculation function loaded successfully")


## 6. Main Simulation Function

The core simulation loop that integrates the reactor equations.


In [ ]:
def run_simulation(gas, config, reactant_info, hf, T_0, p_0, length, diameter, area, roughness, mass_flow_rate, u_0):
    """Run the main simulation loop."""
    n_steps = config['simulation_settings']['n_steps']
    dz = length / n_steps
    
    print(f"Starting simulation with {n_steps} steps...")
    
    # Initialize arrays
    z = np.linspace(0, length, n_steps + 1)
    T = np.zeros(n_steps + 1)
    p = np.zeros(n_steps + 1)
    u = np.zeros(n_steps + 1)
    rho = np.zeros(n_steps + 1)
    Y = np.zeros((n_steps + 1, gas.n_species))
    
    # Set initial conditions
    T[0] = T_0
    p[0] = p_0
    u[0] = u_0
    rho[0] = gas.density
    Y[0, :] = gas.Y
    
    # Create reactor network
    r_vol = area * dz
    r_area = np.pi * diameter
    
    upstream = ct.Reservoir(gas, name='upstream')
    furnace = ct.Reservoir(gas)
    r1 = ct.IdealGasReactor(gas, volume=r_vol)
    w1 = ct.Wall(furnace, r1, A=r_area, Q=hf(0.0), U=0.0, K=0.0)
    downstream = ct.Reservoir(gas, name='downstream')
    
    # Flow controllers
    mfc1 = ct.MassFlowController(upstream, r1, mdot=mass_flow_rate)
    mfc2 = ct.MassFlowController(r1, downstream, mdot=mass_flow_rate)
    
    # Simulation loop
    for i in range(n_steps):
        if i % 20 == 0:
            print(f"Progress: {i}/{n_steps} ({i/n_steps*100:.0f}%)")
        
        # Update heat flux
        dist = z[i]
        w1.heat_flux = hf(dist)
        
        # Calculate pressure drop
        dpdz = calculate_pressure_drop(gas, u[i], diameter, roughness)
        p[i+1] = p[i] - dpdz * dz
        
        # Update gas state
        gas.TPY = T[i], p[i+1], Y[i, :]
        
        # Update reactor
        upstream.syncState()
        r1.syncState()
        
        # Store results
        T[i+1] = gas.T
        u[i+1] = mass_flow_rate / (gas.density * area)
        rho[i+1] = gas.density
        Y[i+1, :] = gas.Y
    
    print("Simulation completed!")
    
    # Create results object
    class SimulationResults:
        def __init__(self, z, T, p, u, rho, Y):
            self.z = z
            self.T = T
            self.P = p
            self.velocity = u
            self.density = rho
            self.Y = Y
    
    return SimulationResults(z, T, p, u, rho, Y)

print("Main simulation function loaded successfully")


## 7. Results Processing and Visualization

Functions to process simulation results and create visualizations.


In [ ]:
def calculate_conversion_and_yields(gas, states, reactant_info):
    """Calculate reactant conversion and product yields."""
    # Get initial and final states
    Y_initial = states.Y[0, :]
    Y_final = states.Y[-1, :]
    
    # Calculate conversion
    feed_species = reactant_info.get('feed_species', '')
    if ':' in feed_species:
        species_name = feed_species.split(':')[0]
    else:
        species_name = feed_species
    
    try:
        species_idx = gas.species_index(species_name)
        conversion = (Y_initial[species_idx] - Y_final[species_idx]) / Y_initial[species_idx] * 100
    except:
        conversion = 0.0
    
    # Calculate yields for key products
    products = ['C2H4', 'CH4', 'H2', 'C2H2', 'C3H6', 'C4H6', 'C4H8', 'C5H8']
    yields = {}
    
    for product in products:
        try:
            product_idx = gas.species_index(product)
            yield_val = Y_final[product_idx] * 100
            yields[product] = yield_val
        except:
            yields[product] = 0.0
    
    return conversion, yields

def create_visualizations(gas, states, config, reactant_info, hf, conversion, yields):
    """Create all visualization plots."""
    # Create fig directory if it doesn't exist
    os.makedirs('fig', exist_ok=True)
    
    # Temperature profile
    plt.figure(figsize=(10, 6))
    plt.plot(states.z, states.T, 'r-', linewidth=2, label='Temperature')
    plt.xlabel('$z$ [m]')
    plt.ylabel('$T$ [K]')
    plt.title('Temperature Profile')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('fig/temperature_profile.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Pressure profile
    plt.figure(figsize=(10, 6))
    plt.plot(states.z, states.P/1e5, 'b-', linewidth=2, label='Pressure')
    plt.xlabel('$z$ [m]')
    plt.ylabel('$p$ [bar]')
    plt.title('Pressure Profile')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('fig/pressure_profile.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Velocity profile
    plt.figure(figsize=(10, 6))
    plt.plot(states.z, states.velocity, 'g-', linewidth=2, label='Velocity')
    plt.xlabel('$z$ [m]')
    plt.ylabel('$u$ [m/s]')
    plt.title('Velocity Profile')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('fig/velocity_profile.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Heat flux profile
    plt.figure(figsize=(10, 6))
    heat_flux_values = [hf(z) for z in states.z]
    plt.plot(states.z, heat_flux_values, 'orange', linewidth=2, label='Heat Flux')
    plt.xlabel('$z$ [m]')
    plt.ylabel('$q$ [W/m²]')
    plt.title('Heat Flux Profile')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('fig/heat_flux_profile.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Product mass fractions
    plt.figure(figsize=(12, 8))
    products = ['C2H4', 'CH4', 'H2', 'C2H2', 'C3H6', 'C4H6', 'C4H8', 'C5H8']
    colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray']
    
    for i, product in enumerate(products):
        try:
            product_idx = gas.species_index(product)
            plt.plot(states.z, states.Y[:, product_idx] * 100, 
                    color=colors[i], linewidth=2, label=product)
        except:
            continue
    
    plt.xlabel('$z$ [m]')
    plt.ylabel('Mass Fraction [%]')
    plt.title('Product Mass Fractions')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('fig/product_mass_fractions.png', dpi=150, bbox_inches='tight')
    plt.show()

print("Results processing and visualization functions loaded successfully")


## 8. Interactive Simulation

Now you can run simulations interactively! Choose your reactant and run the simulation.


In [ ]:
# Choose your reactant here
REACTANT = 'ethane'  # Options: 'ethane', 'propane', 'naphtha', 'n-hexane'

print(f"Initializing simulation for: {REACTANT}")
print("=" * 50)

# Generate configuration
config = generate_configuration(REACTANT, database)
reactant_info = database['reactants'][REACTANT]

print(f"Configuration loaded for {reactant_info['name']} Pyrolysis PFR Simulation")
print(f"Version: {config['simulation_info']['version']}")

# Setup mechanism
gas = setup_mechanism(config)

# Setup initial conditions
T_0, p_0, composition_0, length, diameter, area, roughness, mass_flow_rate, u_0 = setup_initial_conditions(gas, config)

# Setup heat flux
hf, z_profile, heatflux_profile = setup_heat_flux(config)

# Configuration validation
print(f"Configuration validation:")
print(f"  - Mechanism: {config['mechanism']['reaction_mechanism_file']}")
print(f"  - Temperature: {T_0} K")
print(f"  - Pressure: {p_0/1e5:.1f} bar")
print(f"  - Composition: {composition_0}")
print(f"  - Reactant: {reactant_info['name']}")

print("\n" + "=" * 50)


In [ ]:
# Run the simulation
print("Starting simulation...")
start_time = time.time()

states = run_simulation(gas, config, reactant_info, hf, T_0, p_0, length, diameter, area, roughness, mass_flow_rate, u_0)

simulation_time = time.time() - start_time
print(f"Simulation completed in {simulation_time:.2f} seconds")

# Calculate results
conversion, yields = calculate_conversion_and_yields(gas, states, reactant_info)

print(f"\nSIMULATION RESULTS")
print("=" * 30)
print(f"Final temperature: {states.T[-1]:.1f} K")
print(f"Final pressure: {states.P[-1]/1e5:.2f} bar")
print(f"{reactant_info['name']} conversion: {conversion:.1f}%")
print(f"Product yields:")
for product, yield_val in yields.items():
    if yield_val > 0.01:  # Only show significant yields
        print(f"  {product}: {yield_val:.2f}% (mass)")


In [ ]:
# Create visualizations
print("\nCreating visualizations...")
create_visualizations(gas, states, config, reactant_info, hf, conversion, yields)

print("\nAll visualizations created and saved to fig/ directory")


## 9. Export Results

Export simulation results to CSV and create summary files.


In [ ]:
# Export results to CSV
print("Exporting results to CSV...")

# Create results DataFrame
results_data = {
    'z_position_m': states.z,
    'temperature_K': states.T,
    'pressure_Pa': states.P,
    'pressure_bar': states.P / 1e5,
    'velocity_ms': states.velocity,
    'density_kgm3': states.density,
    'heat_flux_Wm2': [hf(z) for z in states.z]
}

# Add all species mass fractions
for i, species in enumerate(gas.species_names):
    results_data[f'{species}_mass_fraction'] = states.Y[:, i]

# Create DataFrame
df = pd.DataFrame(results_data)

# Create results directory if it doesn't exist
os.makedirs('results', exist_ok=True)

# Generate filename
reactant_name = reactant_info['name'].replace(' ', '').replace('-', '')
temp_str = f"{T_0:.0f}"
pressure_str = f"{p_0/1e5:.1f}"
length_str = f"{config['reactor_geometry']['length_m']:.1f}"
diameter_str = f"{config['reactor_geometry']['diameter_m']*1000:.1f}"
massflow_str = f"{config['operating_conditions']['mass_flow_rate_kgps']:.4f}"
steps_str = f"{config['simulation_settings']['n_steps']}"

csv_filename = f'results/results_{reactant_name}_T{temp_str}K_P{pressure_str}bar_L{length_str}m_D{diameter_str}mm_M{massflow_str}kgps_n{steps_str}.csv'
df.to_csv(csv_filename, index=False)
print(f"Saved complete results to {csv_filename}")

# Create summary file
summary_filename = f'results/summary_{reactant_name}_T{temp_str}K_P{pressure_str}bar_L{length_str}m_D{diameter_str}mm_M{massflow_str}kgps_n{steps_str}.dat'

with open(summary_filename, 'w') as f:
    f.write(f"# {reactant_info['name']} Pyrolysis PFR Simulation Summary\n")
    f.write("# =========================================\n")
    f.write(f"# Simulation Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"# Reactant: {reactant_info['name']} ({reactant_info['formula']})\n")
    f.write(f"# Mechanism File: {config['mechanism']['reaction_mechanism_file']}\n")
    f.write(f"# Description: {reactant_info['description']}\n")
    f.write("#\n")
    f.write("# SIMULATION CONFIGURATION\n")
    f.write("# ========================\n")
    f.write(f"# Initial Temperature: {T_0:.1f} K\n")
    f.write(f"# Initial Pressure: {p_0/1e5:.1f} bar\n")
    f.write(f"# Feed Composition: {config['initial_conditions']['composition']}\n")
    f.write(f"# Reactor Length: {config['reactor_geometry']['length_m']:.1f} m\n")
    f.write(f"# Reactor Diameter: {config['reactor_geometry']['diameter_m']*1000:.1f} mm\n")
    f.write(f"# Mass Flow Rate: {config['operating_conditions']['mass_flow_rate_kgps']:.4f} kg/s\n")
    f.write(f"# Number of Integration Steps: {config['simulation_settings']['n_steps']}\n")
    f.write(f"# Solver Tolerance: {config['simulation_settings'].get('solver_tolerance', 1e-6):.2e}\n")
    f.write(f"# Heat Flux File: {config['mechanism']['heat_flux_file']}\n")
    f.write(f"# Initial Heat Flux: {hf(0.0):.0f} W/m²\n")
    f.write(f"# Final Heat Flux: {hf(states.z[-1]):.0f} W/m²\n")
    f.write(f"# Average Heat Flux: {np.mean([hf(z) for z in states.z]):.0f} W/m²\n")
    f.write("#\n")
    f.write("# SIMULATION RESULTS\n")
    f.write("# ==================\n")
    f.write(f"# Final Temperature: {states.T[-1]:.1f} K\n")
    f.write(f"# Temperature Rise: {states.T[-1] - T_0:.1f} K\n")
    f.write(f"# Final Pressure: {states.P[-1]/1e5:.2f} bar\n")
    f.write(f"# Pressure Drop: {(p_0 - states.P[-1])/1e5:.2f} bar\n")
    f.write(f"# Residence Time: {np.trapezoid(1/states.velocity, states.z):.3f} s\n")
    f.write("#\n")
    f.write("# CONVERSION AND YIELDS\n")
    f.write("# =====================\n")
    f.write(f"# {reactant_info['name']} Conversion: {conversion:.1f}%\n")
    for product, yield_val in yields.items():
        f.write(f"# {product} Yield: {yield_val:.2f}% (mass)\n")
    f.write("#\n")
    f.write("# OUTPUT FILES\n")
    f.write("# ============\n")
    f.write(f"# Main Results CSV: {csv_filename}\n")
    f.write(f"# Summary DAT: {summary_filename}\n")
    f.write("#\n")
    f.write("# END OF SUMMARY\n")
    f.write("# ==============\n")

print(f"Saved simulation summary to {summary_filename}")

print("\nSIMULATION COMPLETED SUCCESSFULLY")
print("=" * 50)
print(f"- {reactant_info['name']} conversion: {conversion:.1f}%")
print(f"- Temperature rise: {states.T[-1] - T_0:.1f} K")
print(f"- Pressure drop: {(p_0 - states.P[-1])/1e5:.2f} bar")
print(f"- Residence time: {np.trapezoid(1/states.velocity, states.z):.3f} s")
print(f"- Files generated:")
print(f"  - fig/*.png: Visualization plots")
print(f"  - results/*.csv: Simulation data")
print(f"  - results/*.dat: Simulation summary")
print("=" * 50)


## 10. Parameter Studies

Use this section to run parameter studies by modifying the configuration and re-running simulations.


In [ ]:
# Example: Temperature Study
# Modify the temperature and re-run simulation

temperatures = [900, 925, 950, 975, 1000]  # K
conversions = []
final_temps = []

print("Running temperature study...")
print("=" * 40)

for T_study in temperatures:
    print(f"Running simulation at T = {T_study} K...")
    
    # Create new config with modified temperature
    config_study = config.copy()
    config_study['initial_conditions']['temperature_K'] = T_study
    
    # Setup with new temperature
    gas_study = setup_mechanism(config_study)
    T_0_study, p_0_study, composition_0_study, length_study, diameter_study, area_study, roughness_study, mass_flow_rate_study, u_0_study = setup_initial_conditions(gas_study, config_study)
    hf_study, _, _ = setup_heat_flux(config_study)
    
    # Run simulation
    states_study = run_simulation(gas_study, config_study, reactant_info, hf_study, T_0_study, p_0_study, length_study, diameter_study, area_study, roughness_study, mass_flow_rate_study, u_0_study)
    
    # Calculate results
    conversion_study, _ = calculate_conversion_and_yields(gas_study, states_study, reactant_info)
    
    conversions.append(conversion_study)
    final_temps.append(states_study.T[-1])
    
    print(f"  Conversion: {conversion_study:.1f}%, Final T: {states_study.T[-1]:.1f} K")

# Plot results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(temperatures, conversions, 'ro-', linewidth=2, markersize=8)
plt.xlabel('Initial Temperature [K]')
plt.ylabel('Conversion [%]')
plt.title('Conversion vs Initial Temperature')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(temperatures, final_temps, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Initial Temperature [K]')
plt.ylabel('Final Temperature [K]')
plt.title('Final Temperature vs Initial Temperature')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTemperature Study Results:")
for i, T in enumerate(temperatures):
    print(f"T = {T} K: Conversion = {conversions[i]:.1f}%, Final T = {final_temps[i]:.1f} K")


## 11. Summary

This notebook provides a complete interactive interface for running steam cracking simulations. You can:

### What's Included:
- **Multi-reactant support** (ethane, propane, naphtha, n-hexane)
- **Automatic configuration generation**
- **Interactive parameter adjustment**
- **Real-time visualization**
- **Comprehensive result export**
- **Parameter studies**

### How to Use:
1. **Change Reactant**: Modify the `REACTANT` variable in cell 16
2. **Run Simulation**: Execute cells 16-20 for a complete simulation
3. **Parameter Studies**: Use cell 22 as a template for sensitivity analysis
4. **Custom Analysis**: Add your own analysis cells

### Output Files:
- **fig/**: Visualization plots (PNG format)
- **results/**: Simulation data (CSV) and summaries (DAT)

### Next Steps:
- Try different reactants
- Modify operating conditions
- Run parameter studies
- Add custom analysis functions

---

**Ready for Advanced Chemical Process Simulation**
